## Data source

This project uses publicly available US consumer loan data from LendingClub,
covering loans issued between 2007 and 2015. The dataset contains loan-level
information captured at origination along with subsequent loan outcomes.

Only features available at the time of application are used for modelling to
avoid data leakage. Raw data files are stored locally and excluded from version
control due to size constraints.




In [1]:

import pandas as pd
import numpy as np

df = pd.read_csv("../data/LendingClubloandata.csv", low_memory=False, engine="c")



In [2]:
# Create a stable application ID
df = df.drop(columns=["id","member_id"], errors="ignore")
df = df.reset_index(drop=True)
df["application_id"] = df.index + 1


print("Rows, cols:", df.shape)
print("Loan status counts:\n", df["loan_status"].value_counts(dropna=False).head(20))
df.head(3)

Rows, cols: (2260668, 144)
Loan status counts:
 loan_status
Fully Paid                                             1041952
Current                                                 919695
Charged Off                                             261655
Late (31-120 days)                                       21897
In Grace Period                                           8952
Late (16-30 days)                                         3737
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     31
Name: count, dtype: int64


,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term,application_id
0,2500,2500,2500.0,36 months,13.56,84.92,C,C1,Chef,10+ years,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,1
1,30000,30000,30000.0,60 months,18.94,777.23,D,D2,Postmaster,10+ years,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,2
2,5000,5000,5000.0,36 months,17.97,180.69,D,D1,Administrative,6 years,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,3


In [3]:
# Define GOOD / BAD loan outcomes
good = {
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid"
}

bad = {
    "Charged Off",
    "Default",
    "Late (31-120 days)",
    "Does not meet the credit policy. Status:Charged Off"
}

df = df[df["loan_status"].isin(good | bad)].copy()
df["target_bad"] = df["loan_status"].isin(bad).astype(int)

print("Rows kept:", df.shape[0])
print("Bad rate:", round(df["target_bad"].mean(), 4))


Rows kept: 1328284
Bad rate: 0.2141


In [4]:
# Explicit post-origination leakage columns
leak_cols = [
    "out_prncp", "out_prncp_inv",
    "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
    "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt",
    "next_pymnt_d", "last_credit_pull_d",
    "url", "desc", "title"
]

df = df.drop(columns=[c for c in leak_cols if c in df.columns], errors="ignore")

# Drop hardship / settlement fields
df = df.drop(columns=[c for c in df.columns if c.startswith("hardship_")], errors="ignore")
df = df.drop(columns=[c for c in df.columns if c.startswith("debt_settlement_")], errors="ignore")
df = df.drop(columns=[c for c in df.columns if c.startswith("settlement_")], errors="ignore")

print("Shape after leakage removal:", df.shape)


Shape after leakage removal: (1328284, 110)


In [5]:
# Convert term to numeric
df["term_months"] = (
    df["term"].astype(str)
      .str.extract(r"(\d+)")
      .astype(float)
)

# Clean and parse LendingClub month-year strings 
df["issue_d"] = pd.to_datetime(df["issue_d"].astype(str).str.strip(), format="%b-%Y", errors="coerce")
df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"].astype(str).str.strip(), format="%b-%Y", errors="coerce")

print("issue_d null %:", df["issue_d"].isna().mean())
print("issue_d min/max:", df["issue_d"].min(), df["issue_d"].max())

print("earliest_cr_line null %:", df["earliest_cr_line"].isna().mean())
print("earliest_cr_line min/max:", df["earliest_cr_line"].min(), df["earliest_cr_line"].max())




# Credit history length in months
df["credit_history_mths"] = (
    (df["issue_d"] - df["earliest_cr_line"]).dt.days / 30.4375
)

# Ensure numeric types for key drivers
numeric_cols = [
    "annual_inc", "dti", "revol_util",
    "delinq_2yrs", "inq_last_6mths",
    "open_acc", "total_acc"
]

for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")


issue_d null %: 0.0
issue_d min/max: 2007-06-01 00:00:00 2018-12-01 00:00:00
earliest_cr_line null %: 2.183268036052531e-05
earliest_cr_line min/max: 1934-04-01 00:00:00 2015-09-01 00:00:00


In [7]:
df.to_parquet("../outputs/cleaned.parquet", index=False)
print("Saved cleaned dataset to outputs/cleaned.parquet")


Saved cleaned dataset to outputs/cleaned.parquet


## Data cleaning summary

This notebook prepares a modelling-ready dataset from the raw loan-level
data. Records with ambiguous or incomplete outcomes are removed to ensure
a clearly defined default target. Dates are standardised  and only variables available at origination
are retained.

These steps are critical for building a leakage-safe credit risk model.
Inconsistent outcomes, post-origination variables, or improperly handled
dates can materially bias model performance and lead to unrealistic policy
conclusions. The resulting cleaned dataset provides a stable foundation
for PD modelling and downstream decisioning.
